<div style="background-color: RGB(51,165,182);" >
<h1 style="margin: auto; padding: 20px; color:#fff; ">La poule qui chante</h1>
</div>

##Analyse des groupements de pays pour l'export international

**Contexte**

La poule qui chante est une entreprise française d'agroalimentaire, spécialisée dans l'élevage et la vente de poulets sous le label "Poulet Agriculture Biologique". Son activité est aujourd'hui franco-française, mais le PDG souhaite étudier une expansion à l'international.

**Objectif de la mission**

Proposer une analyse des groupements de pays pouvant être ciblés pour l'export, en s'appuyant sur les données de la FAO, complétées par des données en open data (Banque mondiale, coordonnées géographiques) choisies selon la grille d'analyse PESTEL.

**Objectif de ce notebook**

Ce notebook correspond à la première étape du projet : la préparation et le nettoyage des données. On y regroupe et nettoie plusieurs sources (FAO, Banque mondiale, coordonnées des capitales) pour obtenir un tableau final unique, avec une ligne par pays, prêt à être utilisé dans le notebook 2 (ACP et clustering).

<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 1 - Importation des librairies et chargement des fichiers</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">1.1 - Importation des librairies</h3>
</div>

In [1]:
!pip install geopy

import pandas as pd
import numpy as np
from geopy.distance import geodesic

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">1.2 - Chargements des fichiers</h3>
</div>

In [2]:
drive = True
if drive:
  from google.colab import drive
  drive.mount("/content/drive")

  df_dispo = pd.read_csv("/content/drive/MyDrive/Nadia Openclassrooms/Projet 11/Data/DisponibiliteAlimentaire_2017.csv", encoding='utf-8')
  df_pop = pd.read_csv("/content/drive/MyDrive/Nadia Openclassrooms/Projet 11/Data/Population_2000_2018.csv", encoding='utf-8')
  print("Les fichiers ont été importés via le Drive")
else:
  df_customers = pd.read_csv("customers.csv", sep=';', encoding='utf-8-sig')
  df_products = pd.read_csv("products.csv", sep=';', encoding='utf-8-sig')
  df_transactions = pd.read_csv("Transactions.csv", sep=';', encoding='utf-8-sig')
  print("Les fichiers ont été importés")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Les fichiers ont été importés via le Drive


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 2 - Analyse exploratoire des fichiers</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.1 - Analyse exploratoire du fichier disponibilité alimentaire</h3>
</div>

In [3]:
df_dispo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 176600 entries, 0 to 176599
Data columns (total 14 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   Code Domaine            176600 non-null  object 
 1   Domaine                 176600 non-null  object 
 2   Code zone               176600 non-null  int64  
 3   Zone                    176600 non-null  object 
 4   Code Élément            176600 non-null  int64  
 5   Élément                 176600 non-null  object 
 6   Code Produit            176600 non-null  int64  
 7   Produit                 176600 non-null  object 
 8   Code année              176600 non-null  int64  
 9   Année                   176600 non-null  int64  
 10  Unité                   176600 non-null  object 
 11  Valeur                  176600 non-null  float64
 12  Symbole                 176600 non-null  object 
 13  Description du Symbole  176600 non-null  object 
dtypes: float64(1), int64

In [4]:
print("Doublons:", df_dispo.duplicated().sum())

Doublons: 0


Le fichier est propre, il n'y a aucune valeur manquante et aucun doublon

In [5]:
print(df_dispo.shape)
df_dispo.head()

(176600, 14)


,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole
0,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5511,Production,2511,Blé et produits,2017,2017,Milliers de tonnes,4281.0,S,Données standardisées
1,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5611,Importations - Quantité,2511,Blé et produits,2017,2017,Milliers de tonnes,2302.0,S,Données standardisées
2,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5072,Variation de stock,2511,Blé et produits,2017,2017,Milliers de tonnes,-119.0,S,Données standardisées
3,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5911,Exportations - Quantité,2511,Blé et produits,2017,2017,Milliers de tonnes,0.0,S,Données standardisées
4,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5301,Disponibilité intérieure,2511,Blé et produits,2017,2017,Milliers de tonnes,6701.0,S,Données standardisées


On explore les différents types d'Élément disponibles et la présence d'un produit lié à la volaille.

In [6]:
print(df_dispo['Élément'].nunique(), "éléments différents")
print(df_dispo['Élément'].unique())

17 éléments différents
['Production' 'Importations - Quantité' 'Variation de stock'
 'Exportations - Quantité' 'Disponibilité intérieure'
 'Aliments pour animaux' 'Semences' 'Pertes' 'Résidus' 'Nourriture'
 'Disponibilité alimentaire en quantité (kg/personne/an)'
 'Disponibilité alimentaire (Kcal/personne/jour)'
 'Disponibilité de protéines en quantité (g/personne/jour)'
 'Disponibilité de matière grasse en quantité (g/personne/jour)'
 'Traitement' 'Autres utilisations (non alimentaire)'
 'Alimentation pour touristes']


In [7]:
print(df_dispo['Produit'].nunique(), "produits différents")
produits_volaille = [p for p in df_dispo['Produit'].unique() if 'volaille' in p.lower() or 'poulet' in p.lower()]
print(produits_volaille)

98 produits différents
['Viande de Volailles']


Le fichier contient 17 types d'Élément différents, les plus pertinents pour l'analyse sont :
- Production
- Importations - Quantité
- Exportations - Quantité
- Disponibilité alimentaire en quantité (kg/personne/an)

Il y a 98 produits différents, dont "Viande de Volailles" qui est celui qui nous intéresse.

On va donc filtrer le dataset pour ne garder que ce produit et ces éléments

In [8]:
elements_utiles = ['Production', 'Importations - Quantité', 'Exportations - Quantité', 'Disponibilité alimentaire en quantité (kg/personne/an)']
df_volaille = df_dispo[(df_dispo['Produit'] == 'Viande de Volailles') & (df_dispo['Élément'].isin(elements_utiles))]
print(df_volaille.shape)
df_volaille.head(10)

(645, 14)


,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole
651,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5511,Production,2734,Viande de Volailles,2017,2017,Milliers de tonnes,28.00,S,Données standardisées
652,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,5611,Importations - Quantité,2734,Viande de Volailles,2017,2017,Milliers de tonnes,29.00,S,Données standardisées
658,FBS,Nouveaux Bilans Alimentaire,2,Afghanistan,645,Disponibilité alimentaire en quantité (kg/pers...,2734,Viande de Volailles,2017,2017,kg,1.53,Fc,Donnée calculée
1704,FBS,Nouveaux Bilans Alimentaire,202,Afrique du Sud,5511,Production,2734,Viande de Volailles,2017,2017,Milliers de tonnes,1667.00,S,Données standardisées
1705,FBS,Nouveaux Bilans Alimentaire,202,Afrique du Sud,5611,Importations - Quantité,2734,Viande de Volailles,2017,2017,Milliers de tonnes,514.00,S,Données standardisées
1707,FBS,Nouveaux Bilans Alimentaire,202,Afrique du Sud,5911,Exportations - Quantité,2734,Viande de Volailles,2017,2017,Milliers de tonnes,63.00,S,Données standardisées
1713,FBS,Nouveaux Bilans Alimentaire,202,Afrique du Sud,645,Disponibilité alimentaire en quantité (kg/pers...,2734,Viande de Volailles,2017,2017,kg,35.69,Fc,Donnée calculée
2713,FBS,Nouveaux Bilans Alimentaire,3,Albanie,5511,Production,2734,Viande de Volailles,2017,2017,Milliers de tonnes,13.00,S,Données standardisées
2714,FBS,Nouveaux Bilans Alimentaire,3,Albanie,5611,Importations - Quantité,2734,Viande de Volailles,2017,2017,Milliers de tonnes,38.00,S,Données standardisées
2716,FBS,Nouveaux Bilans Alimentaire,3,Albanie,5911,Exportations - Quantité,2734,Viande de Volailles,2017,2017,Milliers de tonnes,0.00,S,Données standardisées


On transforme les données pour obtenir une seule ligne par pays.

In [9]:
df_volaille_pivot = df_volaille.pivot_table(index='Zone', columns='Élément',values='Valeur').reset_index()
print(df_volaille_pivot.shape)
df_volaille_pivot.head(10)

(172, 5)


Élément,Zone,Disponibilité alimentaire en quantité (kg/personne/an),Exportations - Quantité,Importations - Quantité,Production
0,Afghanistan,1.53,NaN,29.0,28.0
1,Afrique du Sud,35.69,63.0,514.0,1667.0
2,Albanie,16.36,0.0,38.0,13.0
3,Algérie,6.38,0.0,2.0,275.0
4,Allemagne,19.47,646.0,842.0,1514.0
5,Angola,10.56,0.0,277.0,42.0
6,Antigua-et-Barbuda,54.10,0.0,7.0,0.0
7,Arabie saoudite,43.36,10.0,722.0,616.0
8,Argentine,42.24,207.0,8.0,2161.0
9,Arménie,16.06,0.0,35.0,11.0


Des NaN apparaissent pour les pays sans valeur sur certains éléments

In [10]:
print(df_volaille_pivot.isna().sum())

Élément
Zone                                                       0
Disponibilité alimentaire en quantité (kg/personne/an)     0
Exportations - Quantité                                   37
Importations - Quantité                                    2
Production                                                 4
dtype: int64


Ces NaN signifie très probablement que le pays n'a pas ce flux commercial pour la volaille. On remplace donc par 0

In [11]:
df_volaille_pivot = df_volaille_pivot.fillna(0)
print(df_volaille_pivot.isna().sum())

Élément
Zone                                                      0
Disponibilité alimentaire en quantité (kg/personne/an)    0
Exportations - Quantité                                   0
Importations - Quantité                                   0
Production                                                0
dtype: int64


In [12]:
#renommage des colonnes
df_volaille_pivot = df_volaille_pivot.rename(columns={
    'Production': 'prod_volaille_milliers_tonnes',
    'Importations - Quantité': 'import_volaille_milliers_tonnes',
    'Exportations - Quantité': 'export_volaille_milliers_tonnes',
    'Disponibilité alimentaire en quantité (kg/personne/an)': 'dispo_volaille_kg_hab'
})

In [13]:
df_volaille_pivot.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 5 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Zone                             172 non-null    object 
 1   dispo_volaille_kg_hab            172 non-null    float64
 2   export_volaille_milliers_tonnes  172 non-null    float64
 3   import_volaille_milliers_tonnes  172 non-null    float64
 4   prod_volaille_milliers_tonnes    172 non-null    float64
dtypes: float64(4), object(1)
memory usage: 6.8+ KB


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">2.2 - Analyse exploratoire du fichier population</h3>
</div>

In [14]:
df_pop.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4411 entries, 0 to 4410
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Code Domaine            4411 non-null   object 
 1   Domaine                 4411 non-null   object 
 2   Code zone               4411 non-null   int64  
 3   Zone                    4411 non-null   object 
 4   Code Élément            4411 non-null   int64  
 5   Élément                 4411 non-null   object 
 6   Code Produit            4411 non-null   int64  
 7   Produit                 4411 non-null   object 
 8   Code année              4411 non-null   int64  
 9   Année                   4411 non-null   int64  
 10  Unité                   4411 non-null   object 
 11  Valeur                  4411 non-null   float64
 12  Symbole                 4411 non-null   object 
 13  Description du Symbole  4411 non-null   object 
 14  Note                    258 non-null    

In [15]:
print("Doublons:", df_pop.duplicated().sum())

Doublons: 0


Le fichier est propre, il n'y a aucun doublon et aucune valeur manquante pour dans les colonnes qui nous intéressent: 'Zone' et 'Valeur'

In [16]:
print(df_pop.shape)
df_pop.head()

(4411, 15)


,Code Domaine,Domaine,Code zone,Zone,Code Élément,Élément,Code Produit,Produit,Code année,Année,Unité,Valeur,Symbole,Description du Symbole,Note
0,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2000,2000,1000 personnes,20779.953,X,Sources internationales sûres,NaN
1,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2001,2001,1000 personnes,21606.988,X,Sources internationales sûres,NaN
2,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2002,2002,1000 personnes,22600.770,X,Sources internationales sûres,NaN
3,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2003,2003,1000 personnes,23680.871,X,Sources internationales sûres,NaN
4,OA,Séries temporelles annuelles,2,Afghanistan,511,Population totale,3010,Population-Estimations,2004,2004,1000 personnes,24726.684,X,Sources internationales sûres,NaN


In [17]:
print(df_pop['Élément'].unique())
print(df_pop['Année'].min(), "-", df_pop['Année'].max())
print(df_pop['Zone'].nunique(), "pays différents")

['Population totale']
2000 - 2018
238 pays différents


On ne garde que l’année 2017 pour rester cohérent avec les données de disponibilité alimentaire.

In [18]:
df_pop_2017 = df_pop[df_pop['Année'] == 2017][['Zone', 'Valeur']].rename(columns={'Valeur': 'population_milliers'}).reset_index(drop=True)
print(df_pop_2017.shape)
df_pop_2017.head()

(236, 2)


,Zone,population_milliers
0,Afghanistan,36296.113
1,Afrique du Sud,57009.756
2,Albanie,2884.169
3,Algérie,41389.189
4,Allemagne,82658.409


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 3 - Jonction des fichiers</h2>
</div>

<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">3.1 - Fusion disponibilité alimentaire (volaille) + population</h3>
</div>

In [19]:
df_final = df_volaille_pivot.merge(df_pop_2017, on='Zone', how='left')
print(df_final.shape)
df_final.info()

(172, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 6 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Zone                             172 non-null    object 
 1   dispo_volaille_kg_hab            172 non-null    float64
 2   export_volaille_milliers_tonnes  172 non-null    float64
 3   import_volaille_milliers_tonnes  172 non-null    float64
 4   prod_volaille_milliers_tonnes    172 non-null    float64
 5   population_milliers              172 non-null    float64
dtypes: float64(5), object(1)
memory usage: 8.2+ KB


In [20]:
df_final.head()

,Zone,dispo_volaille_kg_hab,export_volaille_milliers_tonnes,import_volaille_milliers_tonnes,prod_volaille_milliers_tonnes,population_milliers
0,Afghanistan,1.53,0.0,29.0,28.0,36296.113
1,Afrique du Sud,35.69,63.0,514.0,1667.0,57009.756
2,Albanie,16.36,0.0,38.0,13.0,2884.169
3,Algérie,6.38,0.0,2.0,275.0,41389.189
4,Allemagne,19.47,646.0,842.0,1514.0,82658.409


In [21]:
#on verifie les valeurs manquantes
print(df_final.isna().sum())

Zone                               0
dispo_volaille_kg_hab              0
export_volaille_milliers_tonnes    0
import_volaille_milliers_tonnes    0
prod_volaille_milliers_tonnes      0
population_milliers                0
dtype: int64


In [22]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 6 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Zone                             172 non-null    object 
 1   dispo_volaille_kg_hab            172 non-null    float64
 2   export_volaille_milliers_tonnes  172 non-null    float64
 3   import_volaille_milliers_tonnes  172 non-null    float64
 4   prod_volaille_milliers_tonnes    172 non-null    float64
 5   population_milliers              172 non-null    float64
dtypes: float64(5), object(1)
memory usage: 8.2+ KB


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">3.2 - Ajout du PIB par habitant</h3>
</div>

Ajout du PIB par habitant issu du site de la Banque mondiale.
Chargement du fichier, puis filtrage sur l’année 2017 pour rester cohérent avec les autres données

In [23]:
df_pib = pd.read_csv("/content/drive/MyDrive/Nadia Openclassrooms/Projet 11/Data/API_NY.GDP.PCAP.PP.CD_DS2_fr_csv_v2_17620.csv", skiprows=4)
print(df_pib.shape)
df_pib.head()

(265, 71)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"PIB par habitant, ($ PPA internationaux courants)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,37524.914920,39287.059713,38554.716783,28733.604511,35207.170909,41863.273996,47521.876547,51712.153136,NaN,NaN
1,NaN,AFE,"PIB par habitant, ($ PPA internationaux courants)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,3843.028379,3727.710785,3804.533834,3707.033612,4022.277659,4359.591407,4496.744898,4635.125049,4840.265200,NaN
2,Afghanistan,AFG,"PIB par habitant, ($ PPA internationaux courants)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,2335.795862,2432.276701,2583.485332,2561.981761,2144.166570,2122.659362,2203.537619,2236.199073,NaN,NaN
3,NaN,AFW,"PIB par habitant, ($ PPA internationaux courants)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,5035.762391,5188.368000,5491.687935,5423.594405,5702.134886,6232.323014,6538.812038,6840.333882,7184.664483,NaN
4,Angola,AGO,"PIB par habitant, ($ PPA internationaux courants)",NY.GDP.PCAP.PP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,8125.706394,8370.228758,8621.659308,7782.151534,8720.187687,9373.408309,9549.793139,9963.590209,10251.345706,NaN


Le fichier de la  Banque mondiale contient des agrégats régionaux en plus des pays. On isole d'abord la colonne 2017.

In [24]:
df_pib_2017 = df_pib[['Country Name', 'Country Code', '2017']].rename(columns={'Country Name': 'Zone', '2017': 'PIB_hab_USD'})
print(df_pib_2017.shape)
print(df_pib_2017.isna().sum())
df_pib_2017.head()

(265, 3)
Zone             2
Country Code     0
PIB_hab_USD     19
dtype: int64


,Zone,Country Code,PIB_hab_USD
0,Aruba,ABW,37524.914920
1,NaN,AFE,3843.028379
2,Afghanistan,AFG,2335.795862
3,NaN,AFW,5035.762391
4,Angola,AGO,8125.706394


In [25]:
print("Il y a", df_pib_2017.duplicated().sum(), "doublon(s)")

Il y a 0 doublon(s)


On supprime les zones et les PIB manquants

In [26]:
df_pib_2017 = df_pib_2017.dropna().reset_index(drop=True)
print(df_pib_2017.isna().sum())

Zone            0
Country Code    0
PIB_hab_USD     0
dtype: int64


Fusion du PIB par habitant avec df_final

On ajoute la variable PIB_hab_USD à notre tableau final. Les agrégats régionaux seront automatiquement exclus car absents de df_final.

In [27]:
df_final = df_final.merge(df_pib_2017, on='Zone', how='left')
print(df_final.shape)
df_final.isna().sum()

(172, 8)


,0
Zone,0
dispo_volaille_kg_hab,0
export_volaille_milliers_tonnes,0
import_volaille_milliers_tonnes,0
prod_volaille_milliers_tonnes,0
population_milliers,0
Country Code,22
PIB_hab_USD,22


On identifie les 22 pays sans PIB pour savoir s’il manque vraiment des données ou si c’est juste un problème de nom entre les fichiers.

In [28]:
pays_sans_pib = df_final[df_final['PIB_hab_USD'].isna()]['Zone']
print(pays_sans_pib)

18                       Bolivie (État plurinational de)
31                              Chine - RAS de Hong-Kong
32                                  Chine - RAS de Macao
33                             Chine, Taiwan Province de
34                                   Chine, continentale
37                                                 Congo
40                                                  Cuba
68                        Iran (République islamique d')
79                                          Kirghizistan
108                                   Nouvelle-Calédonie
120                                  Polynésie française
124    Royaume-Uni de Grande-Bretagne et d'Irlande du...
127                                  République de Corée
128                                République de Moldova
131           République populaire démocratique de Corée
132                          République-Unie de Tanzanie
140                                            Slovaquie
150                            

In [29]:
def trouver_correspondances(liste_pays_manquants, liste_zones_source):
    n = 0
    for pays in liste_pays_manquants:
        matches = []
        for z in liste_zones_source:
            if pays.lower() in z.lower() or z.lower() in pays.lower():
                matches.append(z)
        if matches:
            n += 1
            print(f"{pays} → {matches}")
    print(f"\nIl y a {n} correspondances trouvées sur {len(liste_pays_manquants)} pays")

In [30]:
liste_pays_sans_pib = pays_sans_pib.tolist()
trouver_correspondances(liste_pays_sans_pib, df_pib_2017['Zone'].unique())

Bolivie (État plurinational de) → ['Bolivie']
Chine - RAS de Hong-Kong → ['Chine']
Chine - RAS de Macao → ['Chine']
Chine, Taiwan Province de → ['Chine']
Chine, continentale → ['Chine']
Congo → ['Congo, République démocratique du', 'Congo, République du']
Royaume-Uni de Grande-Bretagne et d'Irlande du Nord → ['Royaume-Uni', 'Irlande']
République de Moldova → ['Moldova']
République-Unie de Tanzanie → ['Tanzanie']
Égypte → ['Égypte, République arabe d’']
États-Unis d'Amérique → ['États-Unis']

Il y a 11 correspondances trouvées sur 22 pays


On renomme 8 pays du fichier df_pib_2017 pour qu’ils correspondent exactement aux noms utilisés par la FAO dans df_final.

In [31]:
corrections_pib = {
    'Bolivie': 'Bolivie (État plurinational de)',
    'Chine': 'Chine, continentale',
    'Congo, République du': 'Congo',
    'Royaume-Uni': "Royaume-Uni de Grande-Bretagne et d'Irlande du Nord",
    'Moldova': 'République de Moldova',
    'Tanzanie': 'République-Unie de Tanzanie',
    'Égypte, République arabe d’': 'Égypte',
    'États-Unis': "États-Unis d'Amérique"}

df_pib_2017['Zone'] = df_pib_2017['Zone'].replace(corrections_pib)

In [32]:
df_final = df_final.drop(columns=['PIB_hab_USD', 'Country Code'])
df_final = df_final.merge(df_pib_2017, on='Zone', how='left')

print(df_final.isna().sum())

Zone                                0
dispo_volaille_kg_hab               0
export_volaille_milliers_tonnes     0
import_volaille_milliers_tonnes     0
prod_volaille_milliers_tonnes       0
population_milliers                 0
Country Code                       14
PIB_hab_USD                        14
dtype: int64


In [33]:
pays_sans_pib_v2 = df_final[df_final['PIB_hab_USD'].isna()]['Zone']
print(pays_sans_pib_v2)

31                       Chine - RAS de Hong-Kong
32                           Chine - RAS de Macao
33                      Chine, Taiwan Province de
40                                           Cuba
68                 Iran (République islamique d')
79                                   Kirghizistan
108                            Nouvelle-Calédonie
120                           Polynésie française
127                           République de Corée
131    République populaire démocratique de Corée
140                                     Slovaquie
150                                      Tchéquie
161        Venezuela (République bolivarienne du)
163                                         Yémen
Name: Zone, dtype: object


Ces pays n’ont pas été détectés par la recherche automatique en raison de leur orthographe. Parmi eux, 5 seront réécrits manuellement.

In [34]:
corrections_pib_2 = {
    'République tchèque': 'Tchéquie',
    'Iran, République islamique d’': "Iran (République islamique d')",
    'République kirghize': 'Kirghizistan',
    'Corée, République de': 'République de Corée',
    'République slovaque': 'Slovaquie',
}

df_pib_2017['Zone'] = df_pib_2017['Zone'].replace(corrections_pib_2)

In [35]:
df_final = df_final.drop(columns=['PIB_hab_USD', 'Country Code'])
df_final = df_final.merge(df_pib_2017, on='Zone', how='left')

print(df_final['PIB_hab_USD'].isna().sum())

9


In [36]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 8 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Zone                             172 non-null    object 
 1   dispo_volaille_kg_hab            172 non-null    float64
 2   export_volaille_milliers_tonnes  172 non-null    float64
 3   import_volaille_milliers_tonnes  172 non-null    float64
 4   prod_volaille_milliers_tonnes    172 non-null    float64
 5   population_milliers              172 non-null    float64
 6   Country Code                     163 non-null    object 
 7   PIB_hab_USD                      163 non-null    float64
dtypes: float64(6), object(2)
memory usage: 10.9+ KB


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">3.3 - Ajout du taux d'urbanisation</h3>
</div>

Ajout de la population urbaine (%) issu du site de la Banque mondiale.
Chargement du fichier, puis filtrage sur l’année 2017 pour rester cohérent avec les autres données

In [37]:
df_pop_urbaine = pd.read_csv("/content/drive/MyDrive/Nadia Openclassrooms/Projet 11/Data/Population_urbaine.csv", skiprows=4)
print(df_pop_urbaine.shape)
df_pop_urbaine.head()

(265, 71)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Population urbaine (% du total),SP.URB.TOTL.IN.ZS,59.023670,59.072800,59.212271,59.432359,59.723340,60.075491,...,62.776849,62.536764,62.288276,62.002884,61.927534,61.881166,61.835149,61.790940,61.748525,NaN
1,NaN,AFE,Population urbaine (% du total),SP.URB.TOTL.IN.ZS,14.640371,14.867730,15.107367,15.353899,15.607152,15.872246,...,35.282987,35.714718,36.097331,36.488322,36.908543,37.360578,37.772301,38.241441,38.702140,NaN
2,Afghanistan,AFG,Population urbaine (% du total),SP.URB.TOTL.IN.ZS,8.122551,8.389595,8.662664,8.942709,9.231134,9.529343,...,24.835282,24.999165,25.143726,25.262210,25.347862,25.393925,25.473053,25.700735,25.940164,NaN
3,NaN,AFW,Population urbaine (% du total),SP.URB.TOTL.IN.ZS,13.946480,14.468419,15.017383,15.592318,16.190616,16.814048,...,48.761009,49.332676,49.905469,50.492321,51.089419,51.700103,52.302109,52.878824,53.477205,NaN
4,Angola,AGO,Population urbaine (% du total),SP.URB.TOTL.IN.ZS,10.387104,10.810545,11.207060,11.581720,11.945180,12.308095,...,65.085053,65.873214,66.663782,67.456827,68.252416,69.050619,69.851503,70.655138,71.461592,NaN


Le fichier de la Banque mondiale contient des agrégats régionaux en plus des pays. On isole d'abord la colonne 2017.

In [38]:
df_pop_urb_2017 = df_pop_urbaine[['Country Name', 'Country Code', '2017']].rename(columns={'Country Name': 'Zone', '2017': 'urbanisation_pct'})
print(df_pop_urb_2017.shape)
print(df_pop_urb_2017.isna().sum())
df_pop_urb_2017.head()

(265, 3)
Zone                2
Country Code        0
urbanisation_pct    1
dtype: int64


,Zone,Country Code,urbanisation_pct
0,Aruba,ABW,62.776849
1,NaN,AFE,35.282987
2,Afghanistan,AFG,24.835282
3,NaN,AFW,48.761009
4,Angola,AGO,65.085053


In [39]:
print("Il y a ", df_pop_urb_2017.duplicated().sum(), "doublon(s)")

Il y a  0 doublon(s)


On supprime les zones et les populations manquantes.

In [40]:
df_pop_urb_2017 = df_pop_urb_2017.dropna().reset_index(drop=True)
print(df_pop_urb_2017.isna().sum())

Zone                0
Country Code        0
urbanisation_pct    0
dtype: int64


Fusion du taux d'urbanisation avec df_final

In [41]:
df_final = df_final.merge(df_pop_urb_2017[['Zone', 'urbanisation_pct']], on='Zone', how='left')

print(df_final.shape)
print(df_final['urbanisation_pct'].isna().sum())

(172, 9)
19


On identifie les 22 pays sans correspondances pour savoir s’il manque vraiment des données ou si c’est juste un problème de nom entre les fichiers.

In [42]:
pays_sans_urba = df_final[df_final['urbanisation_pct'].isna()]['Zone']
print(pays_sans_urba)

18                       Bolivie (État plurinational de)
31                              Chine - RAS de Hong-Kong
32                                  Chine - RAS de Macao
33                             Chine, Taiwan Province de
34                                   Chine, continentale
37                                                 Congo
68                        Iran (République islamique d')
79                                          Kirghizistan
124    Royaume-Uni de Grande-Bretagne et d'Irlande du...
127                                  République de Corée
128                                République de Moldova
131           République populaire démocratique de Corée
132                          République-Unie de Tanzanie
140                                            Slovaquie
150                                             Tchéquie
161               Venezuela (République bolivarienne du)
163                                                Yémen
166                            

Cette liste de pays ressemble aux 22 pays manquants du fichier PIB,  nous allons les comparer.

In [43]:
liste_pays_sans_urba = pays_sans_urba.tolist()
trouver_correspondances(pays_sans_urba, df_pop_urb_2017['Zone'].unique())

Bolivie (État plurinational de) → ['Bolivie']
Chine - RAS de Hong-Kong → ['Chine']
Chine - RAS de Macao → ['Chine']
Chine, Taiwan Province de → ['Chine']
Chine, continentale → ['Chine']
Congo → ['Congo, République démocratique du', 'Congo, République du']
Royaume-Uni de Grande-Bretagne et d'Irlande du Nord → ['Royaume-Uni', 'Irlande']
République de Moldova → ['Moldova']
République-Unie de Tanzanie → ['Tanzanie']
Venezuela (République bolivarienne du) → ['Venezuela']
Yémen → ['Yémen, Rép. du']
Égypte → ['Égypte, République arabe d’']
États-Unis d'Amérique → ['États-Unis']

Il y a 13 correspondances trouvées sur 19 pays


In [44]:
print(liste_pays_sans_pib == liste_pays_sans_urba)

False


La liste de pays sans correspondance étant identique à celle du fichier PIB, on applique le même dictionnaire de correction, puisque les données proviennent du même site et utilisent donc les mêmes noms de pays.

In [45]:
print(pays_sans_pib)

18                       Bolivie (État plurinational de)
31                              Chine - RAS de Hong-Kong
32                                  Chine - RAS de Macao
33                             Chine, Taiwan Province de
34                                   Chine, continentale
37                                                 Congo
40                                                  Cuba
68                        Iran (République islamique d')
79                                          Kirghizistan
108                                   Nouvelle-Calédonie
120                                  Polynésie française
124    Royaume-Uni de Grande-Bretagne et d'Irlande du...
127                                  République de Corée
128                                République de Moldova
131           République populaire démocratique de Corée
132                          République-Unie de Tanzanie
140                                            Slovaquie
150                            

In [46]:
corrections_urba = {**corrections_pib, **corrections_pib_2}
df_pop_urb_2017['Zone'] = df_pop_urb_2017['Zone'].replace(corrections_urba)

In [47]:
df_final = df_final.drop(columns=['urbanisation_pct'])
df_final = df_final.merge(df_pop_urb_2017[['Zone', 'urbanisation_pct']], on='Zone', how='left')

print(df_final['urbanisation_pct'].isna().sum())

6


In [48]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 9 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Zone                             172 non-null    object 
 1   dispo_volaille_kg_hab            172 non-null    float64
 2   export_volaille_milliers_tonnes  172 non-null    float64
 3   import_volaille_milliers_tonnes  172 non-null    float64
 4   prod_volaille_milliers_tonnes    172 non-null    float64
 5   population_milliers              172 non-null    float64
 6   Country Code                     163 non-null    object 
 7   PIB_hab_USD                      163 non-null    float64
 8   urbanisation_pct                 166 non-null    float64
dtypes: float64(7), object(2)
memory usage: 12.2+ KB


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">3.4 - Ajout du score de stabilité politique</h3>
</div>

Ajout du score de stabilité politique sur une échelle de 0 à 100 issu du site de la Banque mondiale. Chargement du fichier, puis filtrage sur l'année 2017

In [49]:
df_stab_politique = pd.read_csv('/content/drive/MyDrive/Nadia Openclassrooms/Projet 11/Data/Stabilité_politique.csv', skiprows=4)
print(df_stab_politique.shape)
df_stab_politique.head(1)

(530, 71)


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,Political Stability - Governance score (0-100),GOV_WGI_PV_SC,NaN,NaN,NaN,NaN,NaN,NaN,...,86.506164,88.683848,88.69726,88.689893,88.660984,88.710743,88.718833,88.6165,NaN,NaN


In [50]:
df_stab_2017 = df_stab_politique[['Country Name', 'Country Code', '2017']].rename(columns={'Country Name': 'Zone', '2017': 'score_stab_politique'})
print(df_stab_2017.shape)
print(df_stab_2017.isna().sum())
df_stab_2017.head()

(530, 3)
Zone                      4
Country Code              0
score_stab_politique    122
dtype: int64


,Zone,Country Code,score_stab_politique
0,Aruba,ABW,86.506164
1,Aruba,ABW,86.506164
2,NaN,AFE,NaN
3,NaN,AFE,NaN
4,Afghanistan,AFG,21.244270


In [51]:
print("Il y a", df_stab_2017.duplicated().sum(), "doublon(s)")

Il y a 265 doublon(s)


In [52]:
#suppression des doublons et des valeurs NaN
df_stab_2017 = df_stab_2017.drop_duplicates().dropna().reset_index(drop=True)
print(df_stab_2017.shape)
print(df_stab_2017.isna().sum())
df_stab_2017.head()

(204, 3)
Zone                    0
Country Code            0
score_stab_politique    0
dtype: int64


,Zone,Country Code,score_stab_politique
0,Aruba,ABW,86.506164
1,Afghanistan,AFG,21.244270
2,Angola,AGO,56.717910
3,Albanie,ALB,68.692869
4,Émirats arabes unis,ARE,78.180968


In [53]:
df_final = df_final.merge(df_stab_2017[['Zone', 'score_stab_politique']], on='Zone', how='left')
print(df_final.shape)
print(df_final['score_stab_politique'].isna().sum())

(172, 10)
21


In [54]:
pays_sans_stab = df_final[df_final['score_stab_politique'].isna()]['Zone']
liste_pays_sans_stab = pays_sans_stab.tolist()
print(liste_pays_sans_stab)

['Bolivie (État plurinational de)', 'Chine - RAS de Hong-Kong', 'Chine - RAS de Macao', 'Chine, Taiwan Province de', 'Chine, continentale', 'Congo', "Iran (République islamique d')", 'Kirghizistan', 'Nouvelle-Calédonie', 'Polynésie française', "Royaume-Uni de Grande-Bretagne et d'Irlande du Nord", 'République de Corée', 'République de Moldova', 'République populaire démocratique de Corée', 'République-Unie de Tanzanie', 'Slovaquie', 'Tchéquie', 'Venezuela (République bolivarienne du)', 'Yémen', 'Égypte', "États-Unis d'Amérique"]


On compare la liste des pays sans correspondance à celle du fichier PIB, pour vérifier si les mêmes pays posent problème.

In [55]:
print(set(liste_pays_sans_pib) - set(liste_pays_sans_stab))

{'Cuba'}


Cuba dispose d'une donnée de stabilité politique dans la source, contrairement
au PIB et à l'urbanisation où le pays était absent. On réutilise donc le même dictionnaire de
corrections.

In [56]:
df_stab_2017['Zone'] = df_stab_2017['Zone'].replace(corrections_urba)

In [57]:
df_final = df_final.drop(columns=['score_stab_politique'])
df_final = df_final.merge(df_stab_2017[['Zone', 'score_stab_politique']], on='Zone', how='left')

In [58]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 172 entries, 0 to 171
Data columns (total 10 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Zone                             172 non-null    object 
 1   dispo_volaille_kg_hab            172 non-null    float64
 2   export_volaille_milliers_tonnes  172 non-null    float64
 3   import_volaille_milliers_tonnes  172 non-null    float64
 4   prod_volaille_milliers_tonnes    172 non-null    float64
 5   population_milliers              172 non-null    float64
 6   Country Code                     163 non-null    object 
 7   PIB_hab_USD                      163 non-null    float64
 8   urbanisation_pct                 166 non-null    float64
 9   score_stab_politique             164 non-null    float64
dtypes: float64(8), object(2)
memory usage: 13.6+ KB


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">3.5 - Suppression des pays sans données complètes</h3>
</div>

In [59]:
print(df_final.shape)
print(df_final.isna().sum())

(172, 10)
Zone                               0
dispo_volaille_kg_hab              0
export_volaille_milliers_tonnes    0
import_volaille_milliers_tonnes    0
prod_volaille_milliers_tonnes      0
population_milliers                0
Country Code                       9
PIB_hab_USD                        9
urbanisation_pct                   6
score_stab_politique               8
dtype: int64


On retire tous les pays ayant au moins un NaN, puisqu'il s'agit des mêmes pays.

In [60]:
df_final = df_final.dropna(subset=['PIB_hab_USD', 'urbanisation_pct', 'score_stab_politique']).reset_index(drop=True)

print(df_final.shape)
print(df_final.isna().sum())

(163, 10)
Zone                               0
dispo_volaille_kg_hab              0
export_volaille_milliers_tonnes    0
import_volaille_milliers_tonnes    0
prod_volaille_milliers_tonnes      0
population_milliers                0
Country Code                       0
PIB_hab_USD                        0
urbanisation_pct                   0
score_stab_politique               0
dtype: int64


<div style="border: 1px solid RGB(51,165,182);" >
<h3 style="margin: auto; padding: 20px; color: RGB(51,165,182); ">3.6 - Ajout de la distance</h3>
</div>

In [61]:
df_distance_fr = pd.read_csv('/content/drive/MyDrive/Nadia Openclassrooms/Projet 11/Data/capitales-du-monde.csv')
print(df_distance_fr.shape)
df_distance_fr.head()

(245, 6)


,CountryName,CapitalName,CapitalLatitude,CapitalLongitude,CountryCode,ContinentName
0,Somaliland,Hargeisa,9.550000,44.050000,NaN,Africa
1,South Georgia and South Sandwich Islands,King Edward Point,-54.283333,-36.500000,GS,Antarctica
2,French Southern and Antarctic Lands,Port-aux-Français,-49.350000,70.216667,TF,Antarctica
3,Palestine,Jerusalem,31.766667,35.233333,PS,Asia
4,Aland Islands,Mariehamn,60.116667,19.900000,AX,Europe


In [62]:
paris = (48.8566, 2.3522)
df_distance_fr['distance_fr_km'] = df_distance_fr.apply(lambda row: geodesic(paris, (row['CapitalLatitude'], row['CapitalLongitude'])).km, axis=1)
df_distance_fr.head()

,CountryName,CapitalName,CapitalLatitude,CapitalLongitude,CountryCode,ContinentName,distance_fr_km
0,Somaliland,Hargeisa,9.550000,44.050000,NaN,Africa,5827.743172
1,South Georgia and South Sandwich Islands,King Edward Point,-54.283333,-36.500000,GS,Antarctica,11997.320087
2,French Southern and Antarctic Lands,Port-aux-Français,-49.350000,70.216667,TF,Antarctica,12671.243210
3,Palestine,Jerusalem,31.766667,35.233333,PS,Asia,3337.392506
4,Aland Islands,Mariehamn,60.116667,19.900000,AX,Europe,1681.930539


In [63]:
df_distance_fr = df_distance_fr[['CountryName', 'CountryCode', 'distance_fr_km']].rename(columns={'CountryName': 'Zone'})
print(df_distance_fr.shape)
print(df_distance_fr.isna().sum())
df_distance_fr.head()

(245, 3)
Zone              0
CountryCode       3
distance_fr_km    0
dtype: int64


,Zone,CountryCode,distance_fr_km
0,Somaliland,NaN,5827.743172
1,South Georgia and South Sandwich Islands,GS,11997.320087
2,French Southern and Antarctic Lands,TF,12671.243210
3,Palestine,PS,3337.392506
4,Aland Islands,AX,1681.930539


In [64]:
#On vérifie les 3 valeurs manquantes
print(df_distance_fr[df_distance_fr.isna().any(axis=1)])

                Zone CountryCode  distance_fr_km
0         Somaliland         NaN     5827.743172
151          Namibia         NaN     8044.945627
240  Northern Cyprus         NaN     2953.653021


La Namibie a bien le CountryCode "NA", mais pandas le transforme en NaN lors de la lecture du CSV.

On corrige donc en réassignant manuellement "NA".

In [65]:
df_distance_fr.loc[df_distance_fr['Zone'] == 'Namibia', 'CountryCode'] = "NA"

In [66]:
print(df_distance_fr[df_distance_fr.isna().any(axis=1)])

                Zone CountryCode  distance_fr_km
0         Somaliland         NaN     5827.743172
240  Northern Cyprus         NaN     2953.653021


In [67]:
#On vérifie les doublons
print("Il y a", df_distance_fr['CountryCode'].duplicated().sum(), "doublon(s)")

Il y a 1 doublon(s)


On remarque que les noms des pays sont en anglais et que le code ISO est en
alpha-2, alors que df_final utilise des noms de pays en français et un code alpha-3. On va utiliser une table
de correspondance alpha-2 et alpha-3 pour faire le lien entre les deux fichiers.

In [68]:
df_code_iso = pd.read_csv('/content/drive/MyDrive/Nadia Openclassrooms/Projet 11/Data/Code_ISO.csv', sep=';')
print(df_code_iso.shape)
df_code_iso.head()

(247, 4)


,Pays,ISO 3166-1 alpha-2,ISO 3166-1 alpha-3,ISO 3166-1 numérique
0,Afghanistan,AF,AFG,4
1,Îles Åland,AX,ALA,248
2,Albanie,AL,ALB,8
3,Algérie,DZ,DZA,12
4,Samoa américaines,AS,ASM,16


In [69]:
#On verifie les valeurs manquantes
print(df_code_iso[df_code_iso.isna().any(axis=1)])

        Pays ISO 3166-1 alpha-2 ISO 3166-1 alpha-3  ISO 3166-1 numérique
152  Namibie                NaN                NAM                   516


Comme pour df_distance_fr, la Namibie est ici aussi considérée comme NaN sur
le code alpha 2.

On corrige donc en réassignant manuellement "NA".

In [70]:
df_code_iso.loc[df_code_iso['Pays'] == 'Namibie', 'ISO 3166-1 alpha-2'] = "NA"

On convertir le code alpha-2 de
df_distance_fr en code alpha-3, afin de pouvoir le fusionner avec df_final qui utilise le code l'alpha-3

In [71]:
df_code_iso = df_code_iso[['ISO 3166-1 alpha-2', 'ISO 3166-1 alpha-3']].rename(columns={'ISO 3166-1 alpha-2': 'CountryCode', 'ISO 3166-1 alpha-3': 'CountryCode_alpha3'})

In [72]:
#Fusion df_distance_fr/df_code_iso
df_distance_fr = df_distance_fr.merge(df_code_iso, on='CountryCode', how='left')

In [73]:
print(df_distance_fr.shape)
print(df_distance_fr['CountryCode_alpha3'].isna().sum())
df_distance_fr.head()

(245, 4)
4


,Zone,CountryCode,distance_fr_km,CountryCode_alpha3
0,Somaliland,NaN,5827.743172,NaN
1,South Georgia and South Sandwich Islands,GS,11997.320087,SGS
2,French Southern and Antarctic Lands,TF,12671.243210,ATF
3,Palestine,PS,3337.392506,PSE
4,Aland Islands,AX,1681.930539,ALA


In [74]:
df_distance_fr = df_distance_fr[['distance_fr_km', 'CountryCode_alpha3']].rename(columns={'CountryCode_alpha3': 'Country Code'})
df_distance_fr.head()

,distance_fr_km,Country Code
0,5827.743172,NaN
1,11997.320087,SGS
2,12671.243210,ATF
3,3337.392506,PSE
4,1681.930539,ALA


In [75]:
#on verifie les valeurs manquantes
print(df_distance_fr[df_distance_fr.isna().any(axis=1)])

     distance_fr_km Country Code
0       5827.743172          NaN
63      7635.352776          NaN
120     1611.544671          NaN
240     2953.653021          NaN


Fusion de df_distance_fr et df_final sur Country Code qui est la clé commune.

In [76]:
df_final = df_final.merge(df_distance_fr, on='Country Code', how='left')

In [77]:
print(df_final.shape)
print(df_final.isna().sum())

(163, 11)
Zone                               0
dispo_volaille_kg_hab              0
export_volaille_milliers_tonnes    0
import_volaille_milliers_tonnes    0
prod_volaille_milliers_tonnes      0
population_milliers                0
Country Code                       0
PIB_hab_USD                        0
urbanisation_pct                   0
score_stab_politique               0
distance_fr_km                     0
dtype: int64


In [78]:
#On vérifie les doublons sur la colonne Zone
print("Doublons sur Zone :", df_final['Zone'].duplicated().sum())

Doublons sur Zone : 0


In [79]:
df_final = df_final.drop(columns=['Country Code'])

In [80]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 163 entries, 0 to 162
Data columns (total 10 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Zone                             163 non-null    object 
 1   dispo_volaille_kg_hab            163 non-null    float64
 2   export_volaille_milliers_tonnes  163 non-null    float64
 3   import_volaille_milliers_tonnes  163 non-null    float64
 4   prod_volaille_milliers_tonnes    163 non-null    float64
 5   population_milliers              163 non-null    float64
 6   PIB_hab_USD                      163 non-null    float64
 7   urbanisation_pct                 163 non-null    float64
 8   score_stab_politique             163 non-null    float64
 9   distance_fr_km                   163 non-null    float64
dtypes: float64(9), object(1)
memory usage: 12.9+ KB


In [81]:
df_final.head()

,Zone,dispo_volaille_kg_hab,export_volaille_milliers_tonnes,import_volaille_milliers_tonnes,prod_volaille_milliers_tonnes,population_milliers,PIB_hab_USD,urbanisation_pct,score_stab_politique,distance_fr_km
0,Afghanistan,1.53,0.0,29.0,28.0,36296.113,2335.795862,24.835282,21.244270,5595.496748
1,Afrique du Sud,35.69,63.0,514.0,1667.0,57009.756,13738.438585,63.331108,59.355008,8649.467089
2,Albanie,16.36,0.0,38.0,13.0,2884.169,14110.672057,55.980276,68.692869,1604.449323
3,Algérie,6.38,0.0,2.0,275.0,41389.189,13493.560749,71.322791,50.016967,1346.107790
4,Allemagne,19.47,646.0,842.0,1514.0,82658.409,54142.741486,80.984254,77.889508,879.241272


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Etape 4 - Export du fichier final</h2>
</div>

In [82]:
print("Nombre de pays :", df_final.shape[0])

population_couverte = df_final['population_milliers'].sum()
population_totale_reference = df_pop_2017['population_milliers'].sum()

pourcentage = (population_couverte / population_totale_reference) * 100
print(f"Population couverte : {pourcentage:.2f}%")

Nombre de pays : 163
Population couverte : 95.95%


Le tableau final, propre et complet, est exporté pour être réutilisé dans
le notebook 2 (ACP et clustering).

In [83]:
df_final.to_csv('donnees_final.csv', index=False)

In [84]:
from google.colab import files
files.download('donnees_final.csv')

Le tableau final contient 163 pays et les colonnes suivantes :

- **Zone** : nom du pays
- **dispo_volaille_kg_hab** : disponibilité alimentaire en volaille, kg/personne/an (FAO)
- **export_volaille_milliers_tonnes** : exportations de volaille, milliers de tonnes (FAO)
- **import_volaille_milliers_tonnes** : importations de volaille, milliers de tonnes (FAO)
- **prod_volaille_milliers_tonnes** : production de volaille, milliers de tonnes (FAO)
- **population_milliers** : population du pays, milliers d'habitants (FAO)
- **PIB_hab_USD** : PIB par habitant en USD (Banque mondiale)
- **urbanisation_pct** : taux d'urbanisation, % de la population totale (Banque mondiale)
- **score_stab_politique** : score de stabilité politique, 0 à 100 (Banque mondiale)
- **distance_fr_km** : distance à vol d'oiseau entre Paris et la capitale du pays (variable annexe, pour un scoring futur)

Les 163 pays représentent 95,95% de la population mondiale.